# Financial News Sentiment Analyzer - EDA

Exploratory Data Analysis of financial news headlines with sentiment analysis and sector classification.

---

## Section 1: Setup & Data Load

Import libraries, load headlines, apply sentiment analysis, and explore data structure.

In [1]:
import os
import sys
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime

# Add parent directory to path for imports
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

# Import analysis functions
from src.fetcher import load_headlines
from src.sentiment import analyze_headlines

print("Loading headlines from cache/API...")
df = load_headlines()
print(f"✓ Raw data loaded: {df.shape[0]} articles, {df.shape[1]} columns")

print("\nApplying sentiment analysis...")
df = analyze_headlines(df)
print(f"✓ Analysis complete")

print(f"\nDataset Shape: {df.shape}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nMissing Values:\n{df.isnull().sum()}")

# Verify required columns
required_columns = ['title', 'source', 'sector', 'sentiment_score', 'sentiment_label']
missing_cols = [col for col in required_columns if col not in df.columns]
if missing_cols:
    print(f"\n⚠️  Missing columns: {missing_cols}")
else:
    print(f"\n✓ All required columns present: {required_columns}")

# Display sample
print(f"\nFirst 3 records:")
print(df[['title', 'source', 'sector', 'sentiment_score', 'sentiment_label']].head(3))

Loading headlines from cache/API...
✓ Raw data loaded: 98 articles, 7 columns

Applying sentiment analysis...
✓ Analysis complete

Dataset Shape: (98, 10)

Data Types:
title                  str
description            str
source                 str
publishedAt            str
category               str
url                    str
fetched_at             str
sentiment_score    float64
sentiment_label        str
sector                 str
dtype: object

Missing Values:
title              0
description        1
source             0
publishedAt        0
category           0
url                0
fetched_at         0
sentiment_score    0
sentiment_label    0
sector             0
dtype: int64

✓ All required columns present: ['title', 'source', 'sector', 'sentiment_score', 'sentiment_label']

First 3 records:
                                               title                   source  \
0  Stock futures are little changed after the S&P...                     CNBC   
1  Former Chick-fil-A emplo

## Section 2: Sentiment Overview

Visualize overall sentiment distribution and average sentiment by sector.

In [2]:
# Sentiment Distribution Pie Chart
sentiment_counts = df['sentiment_label'].value_counts()
fig_pie = px.pie(
    values=sentiment_counts.values, 
    names=sentiment_counts.index,
    title="Sentiment Distribution: Bullish vs Bearish vs Neutral",
    color_discrete_map={'Bullish': '#2ecc71', 'Bearish': '#e74c3c', 'Neutral': '#95a5a6'},
    hole=0.3
)
fig_pie.show()

print(f"\nSentiment Label Counts:")
print(sentiment_counts)
print(f"\nSentiment Percentages:")
print((sentiment_counts / len(df) * 100).round(2))


Sentiment Label Counts:
sentiment_label
Bullish    53
Bearish    28
Neutral    17
Name: count, dtype: int64

Sentiment Percentages:
sentiment_label
Bullish    54.08
Bearish    28.57
Neutral    17.35
Name: count, dtype: float64


In [3]:
# Average Sentiment Score by Sector
sector_sentiment = df.groupby('sector')['sentiment_score'].agg(['mean', 'count']).reset_index()
sector_sentiment = sector_sentiment.sort_values('mean')

fig_bar = px.bar(
    sector_sentiment,
    x='sector',
    y='mean',
    title="Average Sentiment Score by Sector",
    labels={'mean': 'Average Sentiment Score', 'sector': 'Sector'},
    color='mean',
    color_continuous_scale='RdYlGn',
    hover_data={'count': True, 'mean': ':.3f'}
)
fig_bar.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="Neutral")
fig_bar.show()

print(f"\nAverage Sentiment by Sector:")
print(sector_sentiment.to_string(index=False))


Average Sentiment by Sector:
 sector     mean  count
 Energy 0.000000      2
   Tech 0.067577     47
General 0.158018     28
   BFSI 0.231625     20
 Retail 0.911800      1


In [4]:
# Overall Market Mood Summary
overall_sentiment = df['sentiment_score'].mean()
bullish_count = (df['sentiment_label'] == 'Bullish').sum()
bearish_count = (df['sentiment_label'] == 'Bearish').sum()
bullish_pct = (bullish_count / len(df) * 100)

if overall_sentiment >= 0.05:
    mood = "BULLISH 📈"
elif overall_sentiment <= -0.05:
    mood = "BEARISH 📉"
else:
    mood = "NEUTRAL ➡️"

print("="*60)
print("OVERALL MARKET MOOD SUMMARY")
print("="*60)
print(f"Overall Average Sentiment Score: {overall_sentiment:.3f}")
print(f"Market Mood: {mood}")
print(f"Bullish Articles: {bullish_count} ({bullish_pct:.1f}%)")
print(f"Bearish Articles: {bearish_count} ({100-bullish_pct-((df['sentiment_label']=='Neutral').sum()/len(df)*100):.1f}%)")
print(f"Neutral Articles: {(df['sentiment_label']=='Neutral').sum()} ({((df['sentiment_label']=='Neutral').sum()/len(df)*100):.1f}%)")

OVERALL MARKET MOOD SUMMARY
Overall Average Sentiment Score: 0.134
Market Mood: BULLISH 📈
Bullish Articles: 53 (54.1%)
Bearish Articles: 28 (28.6%)
Neutral Articles: 17 (17.3%)


## Section 3: Sector Deep Dive

Analyze sentiment by sector with top positive and negative headlines.

In [5]:
# Grouped Bar Chart: Sentiment Labels Count per Sector
sector_sentiment_counts = df.groupby(['sector', 'sentiment_label']).size().reset_index(name='count')

fig_grouped = px.bar(
    sector_sentiment_counts,
    x='sector',
    y='count',
    color='sentiment_label',
    title="Sentiment Distribution by Sector",
    barmode='group',
    color_discrete_map={'Bullish': '#2ecc71', 'Bearish': '#e74c3c', 'Neutral': '#95a5a6'}
)
fig_grouped.show()

print(f"\nSentiment Counts by Sector:")
print(sector_sentiment_counts.pivot(index='sector', columns='sentiment_label', values='count').fillna(0).astype(int))


Sentiment Counts by Sector:
sentiment_label  Bearish  Bullish  Neutral
sector                                    
BFSI                   4       12        4
Energy                 0        0        2
General                7       16        5
Retail                 0        1        0
Tech                  17       24        6


In [6]:
# Top 3 Most Negative Headlines
print("="*60)
print("TOP 3 MOST NEGATIVE HEADLINES")
print("="*60)
negative_headlines = df.nsmallest(3, 'sentiment_score')[['title', 'source', 'sector', 'sentiment_score']]
for idx, (_, row) in enumerate(negative_headlines.iterrows(), 1):
    print(f"\n{idx}. {row['title'][:80]}...")
    print(f"   Source: {row['source']} | Sector: {row['sector']} | Score: {row['sentiment_score']:.3f}")

# Top 3 Most Positive Headlines
print("\n" + "="*60)
print("TOP 3 MOST POSITIVE HEADLINES")
print("="*60)
positive_headlines = df.nlargest(3, 'sentiment_score')[['title', 'source', 'sector', 'sentiment_score']]
for idx, (_, row) in enumerate(positive_headlines.iterrows(), 1):
    print(f"\n{idx}. {row['title'][:80]}...")
    print(f"   Source: {row['source']} | Sector: {row['sector']} | Score: {row['sentiment_score']:.3f}")

TOP 3 MOST NEGATIVE HEADLINES

1. First-ever 3D view shows how killer T cells destroy cancer - ScienceDaily...
   Source: Science Daily | Sector: Tech | Score: -0.977

2. JPMorgan Exec Claims in Lawsuit Female Boss Drugged Him, Forced Him to Have Sex ...
   Source: TMZ | Sector: Tech | Score: -0.854

3. The most severe Linux threat to surface in years catches the world flat-footed -...
   Source: Ars Technica | Sector: Tech | Score: -0.846

TOP 3 MOST POSITIVE HEADLINES

1. ‘Boo hoo’: The New Yorkers who cheered Mamdani’s ‘tax the rich’ video - Gothamis...
   Source: Gothamist | Sector: Tech | Score: 0.923

2. Free Play Days – Predator: Hunting Grounds, Railway Empire 2, Dragon Ball Fighte...
   Source: Xbox.com | Sector: Tech | Score: 0.915

3. GTA 6 Won't Have Product Placement, Exec Says - GameSpot...
   Source: GameSpot | Sector: Retail | Score: 0.912


## Section 4: Source Analysis

Identify which news sources tend to publish more negative news.

In [7]:
# Source Analysis: Average Sentiment by News Source
source_sentiment = df.groupby('source').agg({
    'sentiment_score': ['mean', 'count'],
    'sentiment_label': lambda x: (x == 'Bullish').sum()
}).reset_index()

source_sentiment.columns = ['source', 'avg_sentiment', 'article_count', 'bullish_count']
source_sentiment = source_sentiment[source_sentiment['article_count'] >= 1].sort_values('avg_sentiment')

# Most negative source
most_negative_source = source_sentiment.iloc[0]
print("="*60)
print("SOURCE WITH MOST NEGATIVE SENTIMENT")
print("="*60)
print(f"Source: {most_negative_source['source']}")
print(f"Average Sentiment Score: {most_negative_source['avg_sentiment']:.3f}")
print(f"Number of Articles: {int(most_negative_source['article_count'])}")
print(f"Bullish Articles: {int(most_negative_source['bullish_count'])}")

# Top 10 sources by average sentiment
print("\n" + "="*60)
print("TOP 10 SOURCES BY AVERAGE SENTIMENT")
print("="*60)
top_10_sources = source_sentiment.tail(10).sort_values('avg_sentiment', ascending=False)
print(top_10_sources[['source', 'avg_sentiment', 'article_count']].to_string(index=False))

SOURCE WITH MOST NEGATIVE SENTIMENT
Source: TMZ
Average Sentiment Score: -0.854
Number of Articles: 1
Bullish Articles: 0

TOP 10 SOURCES BY AVERAGE SENTIMENT
                source  avg_sentiment  article_count
             Gothamist       0.923100              1
              GameSpot       0.911800              1
               Gematsu       0.910000              1
Nintendoeverything.com       0.893400              1
        Pokemon GO Hub       0.831600              1
              Xbox.com       0.810575              4
           Pcguide.com       0.782300              1
          Tipranks.com       0.757900              1
           MarketWatch       0.750600              1
                 Axios       0.718400              1


In [8]:
# Bar Chart: Average Sentiment Score by Top 10 Sources
top_10_for_chart = source_sentiment.tail(10).sort_values('avg_sentiment')

fig_sources = px.bar(
    top_10_for_chart,
    y='source',
    x='avg_sentiment',
    orientation='h',
    title="Average Sentiment Score by Top 10 News Sources",
    labels={'avg_sentiment': 'Average Sentiment Score', 'source': 'News Source'},
    color='avg_sentiment',
    color_continuous_scale='RdYlGn',
    hover_data={'article_count': True, 'avg_sentiment': ':.3f'}
)
fig_sources.add_vline(x=0, line_dash="dash", line_color="gray")
fig_sources.show()

## Section 5: Key Findings

Summary of data-driven insights from the analysis.

In [9]:
# Generate Key Findings Dynamically

# Finding 1: Sector Bullish Percentage (Using Tech as example, but computed dynamically)
tech_data = df[df['sector'] == 'Tech']
if len(tech_data) > 0:
    tech_bullish_pct = (tech_data['sentiment_label'] == 'Bullish').sum() / len(tech_data) * 100
    tech_bullish_count = (tech_data['sentiment_label'] == 'Bullish').sum()
    finding_1 = f"**Tech sector has {tech_bullish_pct:.1f}% bullish sentiment** ({tech_bullish_count} out of {len(tech_data)} articles)"
else:
    finding_1 = "Tech sector data not available"

# Finding 2: Most Bearish Source
bearish_source = source_sentiment.iloc[0]
finding_2 = f"**Most bearish source is {bearish_source['source']}** with an average sentiment score of {bearish_source['avg_sentiment']:.3f}"

# Finding 3: Top Negative Headline
top_negative = df.nsmallest(1, 'sentiment_score').iloc[0]
finding_3 = f"**Top negative headline**: \"{top_negative['title'][:70]}...\" (Score: {top_negative['sentiment_score']:.3f})"

# Print findings
findings_md = f"""## 📊 Key Data-Driven Findings

1. {finding_1}

2. {finding_2}

3. {finding_3}

**Analysis Date**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Total Articles Analyzed**: {len(df)}
**Data Source**: NewsAPI (Business, Technology, Science Categories)
"""

print(findings_md)

# Store findings for markdown cell
key_findings = {
    'finding_1': finding_1,
    'finding_2': finding_2,
    'finding_3': finding_3,
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_articles': len(df)
}

## 📊 Key Data-Driven Findings

1. **Tech sector has 51.1% bullish sentiment** (24 out of 47 articles)

2. **Most bearish source is TMZ** with an average sentiment score of -0.854

3. **Top negative headline**: "First-ever 3D view shows how killer T cells destroy cancer - ScienceDa..." (Score: -0.977)

**Analysis Date**: 2026-05-02 13:13:15
**Total Articles Analyzed**: 98
**Data Source**: NewsAPI (Business, Technology, Science Categories)



## Final Validation and Summary

Confirm all analyses completed successfully.

In [10]:
# Final Validation and Run-All Confirmation
errors = []

# Verify required columns
required_columns = ['title', 'source', 'sector', 'sentiment_score', 'sentiment_label']
for col in required_columns:
    if col not in df.columns:
        errors.append(f"Missing column: {col}")

# Verify non-empty results
if len(df) == 0:
    errors.append("DataFrame is empty")

# Verify sentiment scores are numeric
if not pd.api.types.is_numeric_dtype(df['sentiment_score']):
    errors.append("sentiment_score is not numeric")

# Verify sentiment labels are valid
valid_labels = {'Bullish', 'Bearish', 'Neutral'}
invalid_labels = set(df['sentiment_label'].unique()) - valid_labels
if invalid_labels:
    errors.append(f"Invalid sentiment labels: {invalid_labels}")

# Print validation results
print("="*60)
print("NOTEBOOK EXECUTION VALIDATION")
print("="*60)
if errors:
    print("❌ ERRORS FOUND:")
    for error in errors:
        print(f"  - {error}")
else:
    print("✅ ALL VALIDATIONS PASSED")
    print(f"\n✓ Required columns verified: {required_columns}")
    print(f"✓ DataFrame contains {len(df)} records")
    print(f"✓ All sentiment labels are valid")
    print(f"✓ Sentiment scores are numeric")
    print(f"\n🎉 NOTEBOOK EXECUTION COMPLETE - NO ERRORS")
    print(f"\nTotal Articles Analyzed: {len(df)}")
    print(f"Analysis Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

NOTEBOOK EXECUTION VALIDATION
✅ ALL VALIDATIONS PASSED

✓ Required columns verified: ['title', 'source', 'sector', 'sentiment_score', 'sentiment_label']
✓ DataFrame contains 98 records
✓ All sentiment labels are valid
✓ Sentiment scores are numeric

🎉 NOTEBOOK EXECUTION COMPLETE - NO ERRORS

Total Articles Analyzed: 98
Analysis Timestamp: 2026-05-02 13:13:17


# EDA
Initial exploratory data analysis notebook for financial news sentiment.